# Behavioral Design Patterns
### Strategy | Observer | Command | State | Template Method | Chain | Mediator

> **One coherent system:** ShopFlow -- 500k-user e-commerce platform.

*Run each cell with **Shift + Enter***

## Setup

In [ ]:
from __future__ import annotations
import time
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from collections import defaultdict
from collections.abc import Callable

---
## 1 · Strategy

### Mental Model -- 'The GPS Route Options'

```
WHAT   Define a family of algorithms, encapsulate each one,
       and make them interchangeable at runtime.
WHY    Pricing/sorting/discounting rules change every sprint.
       Strategy = adding a new rule means a new class, zero edits elsewhere.
HOW    Extract varying logic into Strategy classes sharing one interface.
       The Context holds a reference and delegates to the active strategy.
WHEN   Pricing engines | sorting strategies | payment processors |
       notification channels | export formats
```

```
PricingContext
  .strategy = StandardPricing()    <- swap at runtime
  .calculate(cart)
       |
       +-> strategy.calculate(cart)

swap to:
  .strategy = BlackFridayPricing()
  .calculate(cart)                 <- same call, different algorithm
```

In [ ]:
# BEFORE -- pricing logic embedded in checkout

def calculate_price_BAD(cart: dict, user_tier: str, is_black_friday: bool) -> float:
    subtotal = sum(i['price'] * i['qty'] for i in cart['items'])
    if is_black_friday:         discount = subtotal * 0.20
    elif user_tier == 'VIP':    discount = subtotal * 0.15
    elif user_tier == 'B2B':    discount = subtotal * 0.10 if subtotal > 500 else 0
    else:                       discount = 0
    return subtotal - discount
# Every new pricing rule = edit this function = risk breaking all other rules.

In [ ]:
# AFTER -- Strategy Pattern

class PricingStrategy(ABC):
    @abstractmethod
    def calculate(self, subtotal: float, user: dict) -> float:
        # Return the FINAL price after discounts
        ...

class StandardPricing(PricingStrategy):
    def calculate(self, subtotal, user): return subtotal

class BlackFridayPricing(PricingStrategy):
    def calculate(self, subtotal, user): return subtotal * 0.80  # 20% off

class VIPPricing(PricingStrategy):
    def calculate(self, subtotal, user): return subtotal * 0.85  # 15% off

class B2BPricing(PricingStrategy):
    THRESHOLD = 500.0
    def calculate(self, subtotal, user):
        return subtotal * 0.90 if subtotal > self.THRESHOLD else subtotal


class PricingContext:
    def __init__(self, strategy: PricingStrategy) -> None:
        self._strategy = strategy

    def set_strategy(self, s: PricingStrategy) -> None:
        self._strategy = s

    def final_price(self, cart: list[dict], user: dict) -> float:
        subtotal = sum(i['price'] * i['qty'] for i in cart)
        return self._strategy.calculate(subtotal, user)


cart = [{'name': 'Widget', 'price': 50.0, 'qty': 4}]  # subtotal = $200

ctx = PricingContext(StandardPricing())
print(f'Standard:      ${ctx.final_price(cart, {}):.2f}')
ctx.set_strategy(BlackFridayPricing())
print(f'Black Friday:  ${ctx.final_price(cart, {}):.2f}')
ctx.set_strategy(VIPPricing())
print(f'VIP:           ${ctx.final_price(cart, {}):.2f}')
ctx.set_strategy(B2BPricing())
print(f'B2B (<$500):   ${ctx.final_price(cart, {}):.2f}')
# Add new pricing rule -> new class, zero edits to PricingContext or checkout.

### Where This Is Seen in Real Frameworks

| Framework | Strategy usage |
|-----------|---------------|
| **Django REST Framework** | `DEFAULT_AUTHENTICATION_CLASSES` -- swap auth strategies via settings |
| **Celery** | Task routing strategies -- `CELERY_ROUTES` maps tasks to different queues |
| **SQLAlchemy** | Eager vs lazy loading strategies per relationship |
| **Python `sort()`** | `key=` parameter -- the sort strategy is pluggable |
| **scikit-learn** | `Pipeline` steps -- each estimator is a strategy implementing `fit/transform` |

---
## 2 · Observer

### Mental Model -- 'The Newspaper Subscription'

```
WHAT   When one object (subject/publisher) changes state,
       all its dependents (observers/subscribers) are notified automatically.
WHY    Decouples the event source from the event handlers.
       Adding a new handler = add a subscriber, no changes to the publisher.
HOW    Publisher holds a list of subscribers. On event, call each subscriber.
WHEN   Order lifecycle events | domain events | WebSocket notifications |
       audit logs | cache invalidation | plugin systems
```

```
OrderService (Subject)
  |  order_placed event
  +-> EmailSubscriber.handle(event)       -> sends confirmation email
  +-> InventorySubscriber.handle(event)   -> reduces stock
  +-> AnalyticsSubscriber.handle(event)   -> records KPIs
  +-> LoyaltySubscriber.handle(event)     -> awards points

Adding FraudDetection subscriber = zero changes to OrderService.
```

In [ ]:
# AFTER -- Observer / Event Bus Pattern

@dataclass
class OrderEvent:
    event_type: str
    order_id:   str
    payload:    dict = field(default_factory=dict)


class EventBus:
    def __init__(self) -> None:
        self._subscribers: dict[str, list[Callable]] = defaultdict(list)

    def subscribe(self, event_type: str, handler: Callable) -> None:
        self._subscribers[event_type].append(handler)

    def publish(self, event: OrderEvent) -> None:
        for handler in self._subscribers[event.event_type]:
            try:
                handler(event)
            except Exception as e:
                # One subscriber failing does NOT block others
                print(f'  [BUS] {handler.__name__} failed: {e}')


def send_email(event: OrderEvent) -> None:
    print(f'  [Email] Order {event.order_id} confirmation sent')

def reduce_stock(event: OrderEvent) -> None:
    print(f'  [Inventory] Stock reduced for {event.order_id}')

def track_analytics(event: OrderEvent) -> None:
    raise RuntimeError('Analytics service is down!')  # simulated outage

def award_points(event: OrderEvent) -> None:
    print(f'  [Loyalty] Points awarded for {event.order_id}')

def alert_warehouse(event: OrderEvent) -> None:
    print(f'  [Warehouse] Packing {event.order_id}')


bus = EventBus()
for handler in (send_email, reduce_stock, track_analytics, award_points, alert_warehouse):
    bus.subscribe('order_placed', handler)

def place_order(cart: dict, bus: EventBus) -> str:
    order_id = f'ord_{int(time.time())}'
    bus.publish(OrderEvent('order_placed', order_id, {'cart': cart}))
    return order_id


print('Placing order...')
oid = place_order({'email': 'alice@shopflow.com', 'items': [{'sku': 'W1'}]}, bus)
print(f'Order {oid} placed -- checkout succeeded despite analytics outage!')

### Where This Is Seen in Real Frameworks

| Framework | Observer usage |
|-----------|---------------|
| **Django signals** | `post_save.connect(handler, sender=Order)` -- classic GoF Observer |
| **FastAPI / Starlette** | `@app.on_event('startup')` -- lifecycle observers |
| **SQLAlchemy** | `event.listen(mapper, 'after_insert', handler)` |
| **Celery** | Task signals (`task_success`, `task_failure`) |
| **Apache Kafka** | The entire platform -- producers publish events, consumers observe |

---
## 3 · Command

### Mental Model -- 'The Waiter's Order Pad'

```
WHAT   Encapsulate a request (and all its parameters) as an object.
       This enables: queuing, undo/redo, logging, retry.
WHY    'Add to cart' is not just a function call -- it's a record that
       can be undone, replayed, audited, and queued.
HOW    Each action is a Command object with execute() and undo().
       An Invoker stores and runs commands. History enables undo.
WHEN   Undo/redo stacks | job queues | macro recording |
       transactional scripts | audit logs
```

```
AddItemCommand(sku='W1', qty=2)   execute() -> cart=[W1x2]
AddItemCommand(sku='W2', qty=1)   execute() -> cart=[W1x2, W2x1]
RemoveItemCommand(sku='W1')       execute() -> cart=[W2x1]
                                  undo()    -> cart=[W1x2, W2x1]
```

In [ ]:
class CartCommand(ABC):
    @abstractmethod
    def execute(self) -> None: ...
    @abstractmethod
    def undo(self) -> None: ...


@dataclass
class Cart:
    items: dict[str, int] = field(default_factory=dict)
    def add(self, sku: str, qty: int) -> None:
        self.items[sku] = self.items.get(sku, 0) + qty
    def remove(self, sku: str, qty: int) -> None:
        remaining = self.items.get(sku, 0) - qty
        if remaining <= 0: self.items.pop(sku, None)
        else: self.items[sku] = remaining
    def __repr__(self) -> str: return f'Cart({self.items})'


class AddItemCommand(CartCommand):
    def __init__(self, cart: Cart, sku: str, qty: int) -> None:
        self.cart, self.sku, self.qty = cart, sku, qty
    def execute(self) -> None:
        self.cart.add(self.sku, self.qty)
        print(f'  + Added {self.qty}x {self.sku} -> {self.cart}')
    def undo(self) -> None:
        self.cart.remove(self.sku, self.qty)
        print(f'  Undone {self.qty}x {self.sku} -> {self.cart}')


class RemoveItemCommand(CartCommand):
    def __init__(self, cart: Cart, sku: str, qty: int) -> None:
        self.cart, self.sku, self.qty = cart, sku, qty
        self._had: int = 0
    def execute(self) -> None:
        self._had = self.cart.items.get(self.sku, 0)
        self.cart.remove(self.sku, self.qty)
        print(f'  - Removed {self.qty}x {self.sku} -> {self.cart}')
    def undo(self) -> None:
        self.cart.add(self.sku, self._had)
        print(f'  Restored {self.sku} -> {self.cart}')


class CartHistory:
    # Invoker -- runs commands and maintains history for undo
    def __init__(self) -> None: self._history: list[CartCommand] = []
    def execute(self, cmd: CartCommand) -> None:
        cmd.execute(); self._history.append(cmd)
    def undo(self) -> None:
        if self._history: self._history.pop().undo()
        else: print('  Nothing to undo')


cart    = Cart()
history = CartHistory()
history.execute(AddItemCommand(cart, 'Widget', 2))
history.execute(AddItemCommand(cart, 'Gadget', 1))
history.execute(RemoveItemCommand(cart, 'Widget', 1))
print('\nUndo last action:')
history.undo()
print('\nUndo again:')
history.undo()

### Where This Is Seen in Real Frameworks

| Framework | Command usage |
|-----------|______________|
| **Celery** | Each `Task` is a command object -- serialized, queued, retried |
| **Django migrations** | Each `Migration` has `database_forwards` (execute) and `database_backwards` (undo) |
| **SQLAlchemy** | Unit of Work -- collects INSERT/UPDATE/DELETE commands, flushes in one batch |
| **Redis** | `MULTI/EXEC` -- queues commands for atomic execution |
| **Event sourcing** | Each domain event is a command replayed to reconstruct state |

---
## 4 · State

### Mental Model -- 'The Traffic Light'

```
WHAT   An object's behavior changes based on its internal state.
       Each state is its own class -- no giant if/elif chains.
WHY    State machines with if/elif become unmanageable.
       The State pattern makes invalid transitions structurally impossible.
HOW    Each state implements the same interface.
       The context delegates all behavior to the current state object.
WHEN   Order lifecycle | payment states | connection states |
       workflow engines | game entities
```

```
Order State Machine:
  PENDING --pay()--> PAID --ship()--> SHIPPED --deliver()--> DELIVERED
     |                                    |
     +--cancel()--> CANCELLED        cancel()  -- blocked! StateError

  DELIVERED.cancel() raises RuntimeError (not guarded by an if/elif)
```

In [ ]:
class OrderState(ABC):
    @abstractmethod
    def pay(self, order: Order) -> None: ...
    @abstractmethod
    def ship(self, order: Order) -> None: ...
    @abstractmethod
    def deliver(self, order: Order) -> None: ...
    @abstractmethod
    def cancel(self, order: Order) -> None: ...
    def _invalid(self, action: str) -> None:
        raise RuntimeError(f'Cannot {action} from {self.__class__.__name__}')


class Pending(OrderState):
    def pay(self, o):     o.state = Paid();      print(f'  {o.id}: Pending -> Paid')
    def ship(self, o):    self._invalid('ship')
    def deliver(self, o): self._invalid('deliver')
    def cancel(self, o):  o.state = Cancelled(); print(f'  {o.id}: Pending -> Cancelled')

class Paid(OrderState):
    def pay(self, o):     self._invalid('pay')
    def ship(self, o):    o.state = Shipped();   print(f'  {o.id}: Paid -> Shipped')
    def deliver(self, o): self._invalid('deliver')
    def cancel(self, o):  o.state = Cancelled(); print(f'  {o.id}: Paid -> Cancelled')

class Shipped(OrderState):
    def pay(self, o):     self._invalid('pay')
    def ship(self, o):    self._invalid('ship')
    def deliver(self, o): o.state = Delivered(); print(f'  {o.id}: Shipped -> Delivered')
    def cancel(self, o):  self._invalid('cancel')  # cannot cancel shipped order

class Delivered(OrderState):
    def pay(self, o):     self._invalid('pay')
    def ship(self, o):    self._invalid('ship')
    def deliver(self, o): self._invalid('deliver')
    def cancel(self, o):  self._invalid('cancel')

class Cancelled(OrderState):
    def pay(self, o):     self._invalid('pay')
    def ship(self, o):    self._invalid('ship')
    def deliver(self, o): self._invalid('deliver')
    def cancel(self, o):  self._invalid('cancel')


@dataclass
class Order:
    id:    str
    state: OrderState = field(default_factory=Pending)
    def pay(self)     -> None: self.state.pay(self)
    def ship(self)    -> None: self.state.ship(self)
    def deliver(self) -> None: self.state.deliver(self)
    def cancel(self)  -> None: self.state.cancel(self)


order = Order('ORD-001')
order.pay(); order.ship(); order.deliver()
try:
    order.cancel()   # invalid -- already delivered
except RuntimeError as e:
    print(f'  Correctly blocked: {e}')

order2 = Order('ORD-002')
order2.cancel()   # valid from pending

### Where This Is Seen in Real Frameworks

| Framework | State usage |
|-----------|------------|
| **Django** | `order.status` choices + `transition()` (django-fsm library) |
| **Celery** | Task states: PENDING -> STARTED -> SUCCESS / FAILURE / RETRY |
| **asyncio** | Connection states: CLOSED -> CONNECTING -> OPEN -> CLOSING |
| **python-statemachine** | Declarative FSM for Python models |
| **XState (JS)** | Full state chart engine used in React/Vue apps |

---
## 5 · Template Method

### Mental Model -- 'The Recipe Framework'

```
WHAT   Define the skeleton of an algorithm in a base class.
       Subclasses fill in specific steps without changing the overall structure.
WHY    Many workflows share the same shape (validate->process->notify)
       but differ in the details of each step.
HOW    Base class has a final template method that calls abstract hooks.
       Subclasses override the hooks.
WHEN   Data import pipelines | report generators | ETL |
       test setUp/tearDown | HTTP request handlers
```

```
DataImporter.run() [final -- do not override]
  |
  +-> validate(data)    <- abstract -- subclass implements
  +-> transform(data)   <- abstract -- subclass implements
  +-> load(records)     <- abstract -- subclass implements
  +-> notify(count)     <- hook with default (log) -- subclass may override
```

In [ ]:
class DataImporter(ABC):
    # Template: validate -> transform -> load -> notify. Skeleton is fixed.

    def run(self, raw_data: str) -> int:
        # The template method -- algorithm skeleton
        print(f'[{self.__class__.__name__}] Starting import...')
        validated = self.validate(raw_data)
        records   = self.transform(validated)
        count     = self.load(records)
        self.notify(count)
        return count

    @abstractmethod
    def validate(self, raw: str) -> str: ...

    @abstractmethod
    def transform(self, data: str) -> list[dict]: ...

    @abstractmethod
    def load(self, records: list[dict]) -> int: ...

    def notify(self, count: int) -> None:
        # Default hook -- subclass may override
        print(f'  Import complete: {count} records')


class CsvProductImporter(DataImporter):
    def validate(self, raw: str) -> str:
        if 'sku' not in raw.splitlines()[0]: raise ValueError('CSV missing sku header')
        return raw
    def transform(self, data: str) -> list[dict]:
        import io; import csv; return list(csv.DictReader(io.StringIO(data)))
    def load(self, records: list[dict]) -> int:
        for r in records: print(f'  [DB] INSERT product: {r}')
        return len(records)


class JsonOrderImporter(DataImporter):
    def validate(self, raw: str) -> str:
        import json; json.loads(raw); return raw
    def transform(self, data: str) -> list[dict]:
        import json; return json.loads(data)
    def load(self, records: list[dict]) -> int:
        print(f'  [DB] Bulk-insert {len(records)} orders'); return len(records)
    def notify(self, count: int) -> None:
        print(f'  [Slack] {count} orders imported!')   # custom notification


CsvProductImporter().run('sku,name,price\nW1,Widget,9.99\nG2,Gadget,19.99')
print()
JsonOrderImporter().run('[{"id":"ord1","total":49.99},{"id":"ord2","total":99.99}]')

### Where This Is Seen in Real Frameworks

| Framework | Template Method usage |
|-----------|----------------------|
| **Django CBVs** | `View.dispatch()` calls `get()` / `post()` -- subclass overrides the hooks |
| **Django management commands** | `BaseCommand.handle()` is the template; you implement `handle()` |
| **SQLAlchemy** | `TypeDecorator` -- override `process_bind_param` and `process_result_value` |
| **pytest** | `setUp` / `tearDown` -- base test fixture with overridable hooks |
| **Airflow** | `BaseOperator.execute()` -- template with abstract `execute` hook |

---
## 6 · Chain of Responsibility

### Mental Model -- 'The Help Desk Escalation'

```
WHAT   Pass a request along a chain of handlers.
       Each handler decides: handle it, or pass to the next.
WHY    Decouples request sender from receivers.
       Each handler is focused on one concern.
HOW    Each handler has a reference to the next.
WHEN   Middleware pipelines | approval workflows | event filters |
       permission checks | input validation cascades
```

```
Request -> AuthHandler
              | auth OK? -> yes -> RateLimitHandler
                                    | within limit? -> yes -> ValidationHandler
                                                               | -> OK
              At any point a handler can short-circuit the chain.
```

In [ ]:
@dataclass
class ApiRequest:
    token:    str
    user_id:  str
    endpoint: str
    body:     dict = field(default_factory=dict)


class RequestHandler(ABC):
    def __init__(self) -> None: self._next: RequestHandler | None = None

    def set_next(self, h: RequestHandler) -> RequestHandler:
        self._next = h; return h  # fluent chaining

    def handle(self, req: ApiRequest) -> str:
        if self._next: return self._next.handle(req)
        return 'OK: reached the endpoint'


class AuthHandler(RequestHandler):
    VALID = {'tok_admin', 'tok_user1'}
    def handle(self, req: ApiRequest) -> str:
        if req.token not in self.VALID:
            return f'401 Unauthorized: invalid token {req.token!r}'
        print('  [Auth] Authenticated')
        return super().handle(req)


class RateLimitHandler(RequestHandler):
    def __init__(self, limit: int = 3) -> None:
        super().__init__(); self._counts: dict = {}; self.limit = limit
    def handle(self, req: ApiRequest) -> str:
        self._counts[req.user_id] = self._counts.get(req.user_id, 0) + 1
        if self._counts[req.user_id] > self.limit:
            return f'429 Too Many Requests for {req.user_id}'
        print('  [RateLimit] Within limit')
        return super().handle(req)


class ValidationHandler(RequestHandler):
    def handle(self, req: ApiRequest) -> str:
        if req.endpoint == '/checkout' and 'cart' not in req.body:
            return "400 Bad Request: missing 'cart'"
        print('  [Validation] Valid')
        return super().handle(req)


auth  = AuthHandler()
limit = RateLimitHandler(limit=2)
valid = ValidationHandler()
auth.set_next(limit).set_next(valid)

tests = [
    ApiRequest('tok_user1', 'u1', '/checkout', {'cart': [1, 2]}),
    ApiRequest('tok_user1', 'u1', '/checkout', {'cart': [3]}),
    ApiRequest('tok_user1', 'u1', '/checkout', {'cart': [4]}),  # rate limited
    ApiRequest('bad_token',  'u2', '/checkout', {}),            # auth fail
    ApiRequest('tok_admin',  'u3', '/checkout', {}),            # validation fail
]
for req in tests:
    result = auth.handle(req)
    print(f'  -> {result}\n')

### Where This Is Seen in Real Frameworks

| Framework | Chain of Responsibility usage |
|-----------|------------------------------|
| **Django middleware** | `MIDDLEWARE` list -- each middleware calls `get_response(request)` to pass along |
| **FastAPI middleware** | `app.add_middleware()` -- stacked ASGI middleware chain |
| **Python logging** | `Logger.callHandlers()` -- propagates up to parent loggers |
| **WSGI** | Every WSGI app is a chain: each middleware wraps `environ, start_response` |
| **Starlette** | Route matching -- each route tries to match, passes to next if not |